# Portafolio de visualizaciones interpretadas — Delitos en Colombia (2020-2026)

**Actividad 15 híbrida — Representación y métodos gráficos**

**Enfoque social:** este portafolio compara delitos a nivel nacional para identificar líneas de tiempo (2020-2026) y diferencias geográficas entre departamentos y municipios de Colombia, usando datos oficiales de la Policía Nacional filtrados a seis delitos de alto impacto social: violencia intrafamiliar, amenazas, delitos sexuales, homicidio intencional, extorsión y secuestro.

Para cada pregunta analítica se sigue la estructura de interpretación:

> **Hallazgo:** patrón visible en el gráfico.
> **Evidencia:** cifras o elementos del gráfico que lo sustentan.
> **Precaución:** limitación o algo que el gráfico no permite concluir.


In [ ]:
import sys
from pathlib import Path

RAIZ_PROYECTO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.append(str(RAIZ_PROYECTO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.ingesta import cargar_consolidado
from src.graficos import aplicar_estilo

aplicar_estilo()
pd.options.display.float_format = "{:,.2f}".format

df = cargar_consolidado()
df.shape

## Diccionario de datos

El conjunto consolidado (`data/processed/delitos_consolidado.csv`) une, por filas, todos los archivos crudos de `data/raw/delitos/` (2020 a 2026; 2026 es parcial, con datos hasta agosto de 2026). Cada fila es un caso individual, y solo se conservan los delitos definidos en `src/config.DELITOS_FILTRO` — para incluir una categoría adicional basta con agregarla a esa lista.

| Columna | Tipo de variable | Descripción |
|---|---|---|
| `DEPARTAMENTO` | Categórica nominal | Departamento donde ocurrió el hecho (normalizado; 33 valores). |
| `MUNICIPIO` | Categórica nominal | Municipio donde ocurrió el hecho (1.104 valores). |
| `FECHA_HECHO` | Temporal | Fecha del hecho. |
| `AÑO` / `MES` | Temporal | Año y mes derivados de `FECHA_HECHO`. |
| `GENERO` | Categórica nominal | Género de la persona asociada al caso. |
| `GRUPO_EDAD` | Categórica ordinal | Grupo etario de la persona asociada al caso. |
| `TIPO_DELITO` | Categórica nominal | Violencia intrafamiliar, amenaza, delitos sexuales, homicidio intencional, extorsión o secuestro. |
| `ARMAS_MEDIOS` | Categórica nominal | Arma o medio empleado. |
| `CANTIDAD` | Numérica discreta | Número de casos que representa la fila (normalmente 1). |
| `ARCHIVO_ORIGEN` | Identificador | Archivo Excel del que proviene la fila (trazabilidad). |

**Nota de calidad de datos:** durante la construcción del pipeline se corrigieron dos inconsistencias entre archivos de distintos años: nombres de departamento distintos para el mismo lugar (p. ej. "Valle" vs. "Valle del Cauca del Cauca") y Bogotá D.C. etiquetada como "Cundinamarca" en 2020-2024 y como "Bogotá" en 2025-2026 (se unificó por código DANE).

## Pregunta 1 (Distribución): ¿cómo se distribuye la cantidad de casos registrados entre los municipios de Colombia (2020-2026)?

In [ ]:
casos_municipio = (
    df.groupby(["DEPARTAMENTO", "MUNICIPIO"])["CANTIDAD"]
      .sum()
      .reset_index(name="CASOS")
)

valores = casos_municipio["CASOS"]
bins = np.logspace(np.log10(valores.min()), np.log10(valores.max()), 25)

plt.figure(figsize=(9, 5))
plt.hist(valores, bins=bins, color="cornflowerblue", edgecolor="white")
plt.xscale("log")
plt.title("Distribución de casos registrados por municipio (2020-2026)")
plt.xlabel("Casos acumulados por municipio (escala logarítmica)")
plt.ylabel("Número de municipios")
plt.tight_layout()
plt.show()

casos_municipio["CASOS"].describe()

**Hallazgo:** la cantidad de casos está muy concentrada en pocos municipios: la mayoría acumula unos pocos cientos de casos en siete años, mientras un grupo muy reducido —encabezado por Bogotá— concentra varias decenas de miles.

**Evidencia:** de 1.104 municipios, la mediana es 257 casos y el percentil 75 es 603, frente a un máximo de 420.531 (Bogotá D.C.); por esa razón el histograma usa escala logarítmica en el eje X, de lo contrario casi todos los municipios caerían en una sola barra.

**Precaución:** esta concentración también puede reflejar diferencias de población y de capacidad de denuncia/registro entre municipios, no solo diferencias reales en la incidencia delictiva. No debe leerse como "más peligroso" sin normalizar por número de habitantes.

## Pregunta 2 (Comparativa): ¿cómo varía mes a mes la cantidad de casos entre los departamentos con más registros?

In [ ]:
top_departamentos = (
    df.groupby("DEPARTAMENTO")["CANTIDAD"].sum().sort_values(ascending=False).head(6).index.tolist()
)

mensual_depto = (
    df[df["DEPARTAMENTO"].isin(top_departamentos)]
      .groupby(["DEPARTAMENTO", "AÑO", "MES"])["CANTIDAD"]
      .sum()
      .reset_index(name="CASOS_MES")
)

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=mensual_depto,
    x="DEPARTAMENTO",
    y="CASOS_MES",
    order=top_departamentos,
    hue="DEPARTAMENTO",
    legend=False,
)
plt.title("Variabilidad mensual de casos en los 6 departamentos con más registros (2020-2026)")
plt.xlabel("Departamento")
plt.ylabel("Casos por mes")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**Hallazgo:** Bogotá no solo registra más casos en total, sino también la mayor variabilidad mes a mes entre los departamentos con más registros; le siguen Antioquia y Valle del Cauca con niveles y variabilidad intermedios.

**Evidencia:** la caja de Bogotá es notablemente más ancha (mayor rango intercuartílico) que la de Antioquia, Valle del Cauca, Cundinamarca, Santander o Atlántico, con más de 80 meses de observaciones por departamento (2020-2026).

**Precaución:** limitar la comparación a los 6 departamentos con más casos deja fuera al resto del país; además, una mayor variabilidad absoluta puede deberse simplemente al mayor volumen de casos y no a una dinámica delictiva más inestable.

## Pregunta 3 (Dispersión): ¿existe relación entre el número de casos de un departamento en 2020 y en 2025?

In [ ]:
por_anio = df.groupby(["DEPARTAMENTO", "AÑO"])["CANTIDAD"].sum().unstack(fill_value=0)
por_anio.columns = [str(c) for c in por_anio.columns]
por_anio = por_anio.reset_index()

correlacion = por_anio["2020"].corr(por_anio["2025"])

plt.figure(figsize=(8, 8))
sns.scatterplot(data=por_anio, x="2020", y="2025", s=80, alpha=0.8)

limite = max(por_anio["2020"].max(), por_anio["2025"].max()) * 1.05
plt.plot([0, limite], [0, limite], linestyle="--", color="gray", label="Mismo número de casos (2020 = 2025)")

destacados = por_anio[por_anio["DEPARTAMENTO"].isin(["BOGOTA", "ANTIOQUIA", "CÓRDOBA", "ATLÁNTICO"])]
for _, fila in destacados.iterrows():
    plt.annotate(fila["DEPARTAMENTO"], (fila["2020"], fila["2025"]), textcoords="offset points", xytext=(6, 6))

plt.title("Casos por departamento: 2020 vs. 2025 (años completos)")
plt.xlabel("Casos en 2020")
plt.ylabel("Casos en 2025")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Correlación de Pearson: {correlacion:.3f}")

**Hallazgo:** existe una relación muy fuerte y positiva entre el número de casos de un departamento en 2020 y en 2025: el tamaño relativo de cada departamento se mantiene bastante estable en cinco años, aunque no es idéntico.

**Evidencia:** correlación de Pearson de 0.994; Bogotá y Antioquia se separan claramente del resto por su volumen. Algunos departamentos se apartan de la línea de igualdad en ambas direcciones: Atlántico aumentó cerca de 59% entre 2020 y 2025, mientras que Córdoba cayó cerca de 36% y Caldas cerca de 28%.

**Precaución:** la fuerte correlación refleja sobre todo que los departamentos más poblados siguen siendo los más poblados cinco años después; no implica que la situación de seguridad sea estable o comparable año a año, y los casos puntuales de aumento o caída (Atlántico, Córdoba) merecen un análisis aparte antes de sacar conclusiones causales.

## Limitaciones generales del portafolio

- **Sin normalizar por población:** las comparaciones geográficas (departamento/municipio) reflejan volúmenes absolutos, no tasas por habitante; está planeado incorporar población del DANE para calcular tasas por cada 100.000 habitantes.
- **2026 es un año parcial:** los datos de 2026 llegan hasta agosto; no se usan para comparar totales anuales completos (por eso la pregunta 3 usa 2020 vs. 2025).
- **Alcance del delito:** el conjunto de datos corresponde a un subconjunto filtrado de delitos de alto impacto social (violencia intrafamiliar, amenazas, delitos sexuales, homicidio intencional, extorsión, secuestro), no a la totalidad de delitos registrados en Colombia.